# Avaliação de acurácia

Mede aceitação da pessoa certa e rejeição de outras nos limiares de produção. Pré-requisitos e comandos estão no `README.md` desta pasta.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from openhello_eval.calibration import Calibration
from openhello_eval.metrics import format_report
from openhello_eval.workflow import Run

DATA = Path(os.environ.get('OPENHELLO_EVAL_DATA', 'data'))
ENROLL_SHOTS = 3
run = Run(DATA / 'people', DATA / 'embeddings.npz', enroll_shots=ENROLL_SHOTS)
calibration = Calibration.from_file(DATA / 'calibration.bin')
print(len(run.enrolled_names), 'enrolled,', len(run.stranger_names), 'strangers')
print(run.parity(calibration))

## Relatório nos limiares de produção

In [ ]:
result = run.report(calibration)
print(format_report(result))

## Distribuição das pontuações

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(-0.2, 1.0, 120)
ax.hist(result['impostor_scores'], bins=bins, density=True, alpha=0.6, label='outra pessoa')
ax.hist(result['genuine_scores'], bins=bins, density=True, alpha=0.6, label='mesma pessoa')
for row in result['verify']:
    ax.axvline(calibration.threshold(row['target_far'] / ENROLL_SHOTS), color='k', lw=0.6, ls='--')
ax.set_xlabel('similaridade de cosseno'); ax.set_ylabel('densidade'); ax.legend()
plt.show()

## FAR medido contra a meta

Pontos acima da diagonal indicam que o sistema aceita mais impostores do que promete.

In [ ]:
targets = np.array([row['target_far'] for row in result['verify']])
measured = np.array([row['measured_far'] for row in result['verify']])
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.loglog(targets, measured, 'o-', label='medido')
ax.loglog(targets, targets, 'k--', lw=0.8, label='meta')
ax.set_xlabel('FAR alvo'); ax.set_ylabel('FAR medido'); ax.legend()
plt.show()

## Curva ROC empírica

Independe da calibração: mostra o melhor que o modelo permite.

In [ ]:
genuine, impostor = result['genuine_scores'], result['impostor_scores']
cuts = np.quantile(impostor, 1 - np.logspace(-0.3, -np.log10(len(impostor)), 120))
far = np.array([(impostor >= c).mean() for c in cuts])
tar = np.array([(genuine >= c).mean() for c in cuts])
fig, ax = plt.subplots(figsize=(5, 4))
ax.semilogx(far, tar)
ax.set_xlabel('FAR'); ax.set_ylabel('aceitação da pessoa certa'); ax.set_ylim(0.5, 1.0); ax.grid(alpha=0.3)
plt.show()